In [5]:
from PIL import Image, ImageDraw, ImageFilter
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import LinearLocator
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import Image as IMG
from IPython.display import display
import imageio
import os

#supresses showing plots, instead load generated png files
matplotlib.use('Agg')

#　Default generates plots in dark mode
plt.style.use('dark_background') 
nice_green = (0.3,0.6,0.1)
nice_cyan = (0,0.8,0.6)
nice_blue = (0.3,0.4,1)
nice_purple = (1,0.1,1)
nice_red = (0.8,0,0.1)
nice_yellow = (1,0.7,0)
nice_orange = (1,0.5,0)
off_white = (1,0.9,1)
off_black = (0.2,0.2,0.2)
black = (0,0,0)

nice_lightgray = (0.7,0.7,0.7)
nice_darkgray = (0.2,0.2,0.2)

off_white = (1,0.9,1)

In [6]:
# generate non linearly separabe datapoints in 2D
N_A = 300 
N_B = 200
r = 5

batch = np.random.uniform(-10, 10, size=(N_A, 2))
mask = np.linalg.norm(batch, axis=1) >= r
A = batch[mask][:N_A]

batch = np.random.uniform(-10, 10, size=(N_B, 2))
mask = np.linalg.norm(batch, axis=1) < r
B = batch[mask][:N_B]

In [7]:

def generate_projection_gif(A, B, filename="projection_loop_one_class.gif", n_frames=120, fps=10):
    # Angles for smooth looping (0 -> 2pi)
    angles = np.linspace(0, 2*np.pi, n_frames)
    images = []

    xlim = ylim = (-10, 10)
    line_len = 12
    
    for theta in angles:
        # Projection vector
        v = np.array([np.cos(theta), np.sin(theta)])
        # Project data onto line
        A_proj = A @ v
        B_proj = B @ v

        fig, (ax_scatter, ax_hist) = plt.subplots(1, 2, figsize=(14, 6))
        plt.style.use('dark_background')
        
        # Scatter plot
        ax_scatter.set_xlim(xlim)
        ax_scatter.set_ylim(ylim)
        ax_scatter.set_xlabel("$x_1$")
        ax_scatter.set_ylabel("$x_2$")
        ax_scatter.scatter(A[:,0], A[:,1], color=(0.2,0.4,1), s=60, label="A")
        
        # Rotating line
        ax_scatter.plot([-line_len*v[0], line_len*v[0]], 
                        [-line_len*v[1], line_len*v[1]], color=nice_darkgray, lw=2)

        # Add number line along the red projection axis
        ticks = np.arange(-line_len, line_len+1, 2)
        for t in ticks:
            point = t*v
            ax_scatter.plot([point[0]-0.1*v[1], point[0]+0.1*v[1]],
                            [point[1]+0.1*v[0], point[1]-0.1*v[0]], color=nice_darkgray, lw=1)
            ax_scatter.text(point[0]+0.2*v[1], point[1]-0.2*v[0], f"{t:.0f}", color=nice_lightgray, fontsize=14)

        ax_scatter.legend()
        ax_scatter.set_title("Scatter plot with projection line")
        
        # Histogram of projections
        ax_hist.hist(A_proj, bins=30, alpha=1, color=(0.2,0.4,1), edgecolor='black', label='A')
        ax_hist.set_xlim(-10, 10)
        ax_hist.set_xlabel("Projection onto line")
        ax_hist.set_ylabel("Frequency")
        ax_hist.legend()
        ax_hist.set_title(f"Histogram along line angle={np.degrees(theta):.1f}°")
        
        # Save frame
        fig.canvas.draw()
        image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
        image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        images.append(image)
        plt.close(fig)

    # Make it loop smoother by reversing frames
    loop_images = images 
    imageio.mimsave(filename, loop_images, fps=fps, loop=0)

# Example usage:
generate_projection_gif(A, B)


/var/folders/0q/699d2ftx6y1g94yr56ph98h40000gn/T/ipykernel_5158/1463634238.py:51: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
/var/folders/0q/699d2ftx6y1g94yr56ph98h40000gn/T/ipykernel_5158/1463634238.py:51: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
/var/folders/0q/699d2ftx6y1g94yr56ph98h40000gn/T/ipykernel_5158/1463634238.py:51: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
/var/folders/0q/699d2ftx6y1g94yr56ph98h40000gn/T/ipykernel_5158/1463634238.py:51: MatplotlibDeprecationWarning: The tostring_rg

In [8]:
def phi_map_2d_to_3d(X):
    X = np.asarray(X)
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = x1**2 + x2**2
    return np.column_stack([x1, x2, x3])

In [9]:
def make_falling_2_class_3D_gif(
        A, B, 
        gif_name="falling.gif", 
        folder="frames", 
        n_frames=100, 
        rotation_factor=0.25,  # fraction of full 360° rotation per frame
        reverse=False           # True: points rise back up
    ):
    import os, imageio
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D
    
    os.makedirs(folder, exist_ok=True)

    Ax, Ay, Az = A[:,0], A[:,1], A[:,2]
    Bx, By, Bz = B[:,0], B[:,1], B[:,2]

    filenames = []

    # define z start and end
    zA_start, zA_end = Az.copy(), np.zeros_like(Az)
    zB_start, zB_end = Bz.copy(), np.zeros_like(Bz)

    if reverse:
        zA_start, zA_end = zA_end, zA_start
        zB_start, zB_end = zB_end, zB_start

    for i in range(n_frames):
        fig = plt.figure(figsize=(8,6))
        ax = fig.add_subplot(111, projection='3d')
        plt.style.use('dark_background')

        # falling faster: t runs from 0 to 1 in first half of frames
        t = min(i / (n_frames//2), 1.0)

        Az_interp = (1-t)*zA_start + t*zA_end
        Bz_interp = (1-t)*zB_start + t*zB_end

        ax.scatter(Ax, Ay, Az_interp, s=30, color=nice_blue, edgecolor=(0.2,0.2,0.8))
        ax.scatter(Bx, By, Bz_interp, s=40, color=nice_orange, marker='*', edgecolor=nice_yellow)

        # axes labels
        ax.set_xlabel(r"$x_1$", fontsize=14)
        ax.set_ylabel(r"$x_2$", fontsize=14, rotation=90)
        ax.set_zlabel(r"$x_3$", fontsize=14, color=(0.5,0.5,0.5), rotation=0)

        # fixed limits
        xlim = (min(Ax.min(), Bx.min()), max(Ax.max(), Bx.max()))
        ylim = (min(Ay.min(), By.min()), max(Ay.max(), By.max()))
        zlim = (0, max(Az.max(), Bz.max()))
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_zlim(*zlim)

        # dark mode styling
        black = (0,0,0)
        off_black = (0.05,0.05,0.05)
        ax.xaxis.pane.set_facecolor(black)
        ax.yaxis.pane.set_facecolor(black)
        ax.zaxis.pane.set_facecolor(off_black)  
        ax.xaxis.pane.set_edgecolor(black)
        ax.yaxis.pane.set_edgecolor(black)
        ax.zaxis.pane.set_edgecolor(off_black)
        ax.xaxis._axinfo["grid"]['color'] = (0.5,0.5,0.5)
        ax.yaxis._axinfo["grid"]['color'] = (0.5,0.5,0.5)
        ax.zaxis._axinfo["grid"]['color'] = (0.5,0.5,0.5)
        ax.tick_params(axis='z', colors='gray')
        ax.zaxis.line.set_color('black') 

        # slower rotation: multiply i by rotation_factor*360°
        azim = rotation_factor * 360 * i / n_frames
        ax.view_init(elev=30, azim=azim)

        # save frame
        fname = f"{folder}/frame_{i:03d}.png"
        plt.savefig(fname, dpi=120)
        plt.close(fig)
        filenames.append(fname)

    # make GIF
    with imageio.get_writer(gif_name, mode='I', duration=0.04, loop=0) as writer:
        for fname in filenames:
            writer.append_data(imageio.imread(fname))


A3 = phi_map_2d_to_3d(A)
B3 = phi_map_2d_to_3d(B)
make_falling_2_class_3D_gif(A3, B3, gif_name="falling.gif")

/var/folders/0q/699d2ftx6y1g94yr56ph98h40000gn/T/ipykernel_5158/3410253219.py:83: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(fname))
